### 1. Підготовка ознак та стратифікований поділ даних
Кодування категоріальних колонок числовими індексами та розбиття вибірки на навчальну (80%) і тестову (20%) зі збереженням пропорції класів (`stratify=y`).

In [1]:
import time
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_parquet("flights_clean.parquet")

cat_cols = ["AIRLINE", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT"]
for col in cat_cols:
  df[col] = df[col].astype("category").cat.codes

features = [
    "MONTH",
    "DAY_OF_WEEK",
    "dep_hour",
    "DISTANCE",
    "AIRLINE",
    "ORIGIN_AIRPORT",
    "DESTINATION_AIRPORT",
]
X = df[features]
y = df["is_delayed"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train розмір: {X_train.shape[0]} рядків")
print(f"Test розмір: {X_test.shape[0]} рядків")

Train розмір: 4571206 рядків
Test розмір: 1142802 рядків


### 2. Навчання Baseline (простого рішення) та Logistic Regression
- Побудова наївного класифікатора за найчастішим класом (`DummyClassifier`).
- Стандартизація числових ознак за допомогою `StandardScaler`.
- Навчання лінійної моделі `LogisticRegression` та оцінка результатів на тестових даних.

In [2]:
results = []

dummy = DummyClassifier(strategy="most_frequent")
t0 = time.time()
dummy.fit(X_train, y_train)
t_dummy = round(time.time() - t0, 2)

y_pred_dummy = dummy.predict(X_test)
results.append({
    "Модель": "Baseline (Most Frequent)",
    "Час навчання (с)": t_dummy,
    "Accuracy": round(accuracy_score(y_test, y_pred_dummy), 4),
    "F1-Score": round(f1_score(y_test, y_pred_dummy, zero_division=0), 4),
    "ROC-AUC": round(roc_auc_score(y_test, dummy.predict_proba(X_test)[:, 1]), 4),
})

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_reg = LogisticRegression(max_iter=200, random_state=42)
t0 = time.time()
log_reg.fit(X_train_scaled, y_train)
t_lr = round(time.time() - t0, 2)

y_pred_lr = log_reg.predict(X_test_scaled)
y_prob_lr = log_reg.predict_proba(X_test_scaled)[:, 1]

results.append({
    "Модель": "Logistic Regression",
    "Час навчання (с)": t_lr,
    "Accuracy": round(accuracy_score(y_test, y_pred_lr), 4),
    "F1-Score": round(f1_score(y_test, y_pred_lr, zero_division=0), 4),
    "ROC-AUC": round(roc_auc_score(y_test, y_prob_lr), 4),
})

print("Baseline та Logistic Regression навчено.")

Baseline та Logistic Regression навчено.


### 3. Навчання моделі градієнтного бустингу (LightGBM) та фінальне порівняння
- Тренування градієнтного бустингу `LGBMClassifier` на повному тренувальному наборі.
- Порівняння всіх трьох моделей за метриками Accuracy, F1-Score, ROC-AUC та часом навчання на нових даних.

In [4]:
lgb_model = lgb.LGBMClassifier(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)

t0 = time.time()
lgb_model.fit(X_train, y_train)
t_lgb = round(time.time() - t0, 2)

y_pred_lgb = lgb_model.predict(X_test)
y_prob_lgb = lgb_model.predict_proba(X_test)[:, 1]

results.append({
    "Модель": "LightGBM",
    "Час навчання (с)": t_lgb,
    "Accuracy": round(accuracy_score(y_test, y_pred_lgb), 4),
    "F1-Score": round(f1_score(y_test, y_pred_lgb), 4),
    "ROC-AUC": round(roc_auc_score(y_test, y_prob_lgb), 4),
})

df_results = pd.DataFrame(results)
print(df_results.to_markdown(index=False))
df_results

| Модель                   |   Час навчання (с) |   Accuracy |   F1-Score |   ROC-AUC |
|:-------------------------|-------------------:|-----------:|-----------:|----------:|
| Baseline (Most Frequent) |               0.19 |     0.823  |     0      |    0.5    |
| Logistic Regression      |               1.58 |     0.823  |     0      |    0.6366 |
| LightGBM                 |              24.36 |     0.8235 |     0.0148 |    0.6916 |
| LightGBM                 |              22.04 |     0.8235 |     0.0148 |    0.6916 |


,Модель,Час навчання (с),Accuracy,F1-Score,ROC-AUC
0,Baseline (Most Frequent),0.19,0.8230,0.0000,0.5000
1,Logistic Regression,1.58,0.8230,0.0000,0.6366
2,LightGBM,24.36,0.8235,0.0148,0.6916
3,LightGBM,22.04,0.8235,0.0148,0.6916
